# Langextract aplication with atributes

In [1]:
import os
import re
import timeit
import random
import textwrap
import pandas as pd
import langextract as lx
from rich.pretty import pprint
from dotenv import load_dotenv
from more_itertools import unique_justseen
from aymurai.database.utils import text_to_uuid
from aymurai.api.endpoints.routers.misc.document_extract import extraction

In [3]:
MAIN_DF ='documents0203entrerios-docs02-08-sin05.csv' #'df-NER-vals-02-08-sin05.csv'
DOCS_PATH = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/'#'/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' #'/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' #

In [4]:
df = pd.read_csv(MAIN_DF)
df.head()

,text,prediction,validation,id_x,created_at_x,updated_at,id_y,document_id,paragraph_id,order,created_at_y,name
0,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,57f85770879c420d8de2a0f3267b653b,518a7f34ad865b91ad95d954093f091a,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-22 21:20:53-03:00,document-02.docx
1,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,768db7897dc943489e9570a117975fe1,2536b5d2111d55c6b545e6bcbb75e002,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-27 01:00:28-03:00,document-08.docx
2,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,2c74793eac2341969aa3dcce7950f43b,6a7d2422da6a5852b68a7bee678d1aae,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-27 02:56:14-03:00,document-04.docx
3,ANTECEDENTES,[],[],adee5f8144eb5e78abb3c56121e906cd,2025-08-08 19:40:02-03:00,2025-08-08 19:40:45-03:00,b31e11b3cff24069b7cf3faa666fbdec,0aec75365ad0511698f72bacb8b88212,adee5f8144eb5e78abb3c56121e906cd,NaN,2025-08-22 19:41:55-03:00,document-03.docx
4,ANTECEDENTES,[],[],adee5f8144eb5e78abb3c56121e906cd,2025-08-08 19:40:02-03:00,2025-08-08 19:40:45-03:00,4abeb6f8ff3340cb99cfe61b16276012,518a7f34ad865b91ad95d954093f091a,adee5f8144eb5e78abb3c56121e906cd,NaN,2025-08-22 21:20:53-03:00,document-02.docx


In [5]:
set(df.name)

{'aymurai - ejemplo 02.docx',
 'aymurai - ejemplo 03.docx',
 'document-02.docx',
 'document-03.docx',
 'document-04.docx',
 'document-06.docx',
 'document-07.docx',
 'document-08.docx',
 nan}

## Prompt & example definitions

In [21]:
PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles y ENTIDADES RELEVANTES en documentos judiciales en español.
Devolvés SOLO spans EXACTOS del texto (sin parafrasear). Si una clase no aparece, no devuelvas nada.
No inventes códigos ni completes por contexto. No superpongas entidades (una mención = una extracción).

REGLAS GENERALES
- Aceptá mayúsculas, acentos faltantes, comillas y formatos no estándar si el span es claro.
- Las iniciales de personas cuentan como PER (p. ej., “M.F.R.”).
- Si un nombre aparece con títulos/cargos delante (p. ej., “Sra. Jueza de Familia Dra. Gabriela Montiel”),
  extraé PER solo con el nombre completo (“Gabriela Montiel”) y, si corresponde, agregá ROL_PROCESAL=Juez.
- Si una entidad aparece varias veces, extraela cada vez (spans exactos).

CARÁTULA Y ENCABEZADOS
- En carátulas del tipo: “Apellido, Nombre c/ Apellido, Nombre s/ …” EXTRAÉ:
  • PER: el bloque “Apellido, Nombre” ANTES de “c/”.
  • PER: el bloque “Apellido, Nombre” DESPUÉS de “c/” y ANTES de “s/”.
  Conservá comas y el orden exacto.
- NUM_EXPEDIENTE: extraé \d+/\d{4} cuando aparezca con cualquiera de estas variantes textuales
  (aceptá puntos y espacios variables): “Expediente”, “Expte.”, “Expte”, “Exp.”, “Exp. N°/Nº/N.o”, “Expte. N°/Nº/N.o”.

AGRUPAMIENTO (group_index)
- Uní la misma persona (PER) con sus atributos: DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO,
  EDAD, NACIONALIDAD, NUM_MATRICULA. Ej.: “Ana Carolina Rodríguez, DNI 34.112.456” → mismo group_index.

DIRECCION
- Formatos válidos: calle + altura (“9 de Julio 123”), intersección (“Callao y Corrientes”), o número suelto.
- Atributo DIRECCION.TIPO solo si está explícito (domicilio_real, domicilio_legal, domicilio_laboral, etc.).

LOC
- Localidad / provincia / país, inclusive con prefijos “CIUDAD DE …”, “Ciudad de …”, “Provincia de …”
  y en MAYÚSCULAS. No extraigas instituciones (p. ej., “Juzgado de …”) como LOC.

RELACION (simplificada)
- Usá RELACION solo para medidas/vínculos claros. Sttributes posibles:
  - TIPO ∈ {"prohibicion_acercamiento","cese_perturbacion","contacto_prohibido","orden_cautelar"}
  - SUJETO_ACTIVO_GROUP (int), SUJETO_PASIVO_GROUP (int)
- Mantener spans breves y literales.

CLASES
- PER, DNI (##.###.### o 7–8 dígitos), CUIT_CUIL (##-########-#), CUIJ (formato usual),
  NUM_EXPEDIENTE (\d+/\d{4}), NUM_ACTUACION, DIRECCION, TELEFONO, CORREO_ELECTRONICO,
  LOC, EDAD, NACIONALIDAD, ESTUDIOS, LINK, PATENTE_DOMINIO, MARCA_AUTOMOVIL,
  NUM_CAJA_AHORRO, CBU, NUM_MATRICULA, FECHA (solo fechas explícitas; no “ayer”).
""")


In [18]:
LONG_PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles y de ENTIDADES RELEVANTES para análisis en documentos judiciales en español. 
Leé el documento y extrae SOLO spans exactos (sin parafrasear ni inferir) de las clases definidas más abajo.

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como PER (no confundir con siglas de otras cosas).
- Extraé SOLO spans EXACTOS que estén en el texto.
- Si una clase NO aparece, no devuelvas nada de esa clase.
- No inventes códigos ni números; no completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- Si una entidad está mal escrita, incompleta, repetida o en un formato no estándar, como ser entre comillas o dentro de una carátura, pero claramente corresponde, extraela igual.
- Si una entidad no está clara, no la extraigas.

AGRUPAMIENTO:
- Cada persona (PER) se agrupa con sus atributos en el mismo group_index: DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA.
- En PER podés usar atributo ROL_PROCESAL solo si está explícito en el texto. 
   Valores posibles de ROL_PROCESAL:
   "Juez", "Fiscal", "Secretario", "Prosecretario", "Mediador", "Asesor Tutelar", "Imputado", "Acusado", 
   "Victima", "Damnificado", "Denunciante", "Querellante", "Actor", "Demandado", "Testigo", 
   "Perito", "Defensor Oficial", "Defensor Particular", "Apoderado", "Tutor", "Curador", "Policia", 
   "Perjudicado", "Beneficiario".
- En DIRECCION podés usar atributo TIPO solo si está indicado en el texto. Valores posibles:
   "domicilio_real", "domicilio_legal", "domicilio_laboral", "domicilio_de_la_victima", "domicilio_del_imputado","domicilio_del_testigo"
- En PER podés usar atributo TIPO solo si está indicado en el texto. Valores posibles:
   "conyugue_de_la_victima", "hijo/a_de_la_victima", etc
- Una DIRECCION puede expresarse como: 
   - calle + altura ("9 de Julio 123"),
   - intersección de calles ("Callao y Corrientes"),
   - número de domicilio aislado.

RELACIONES INTER-GRUPOS:
- Usá la clase RELACION para conectar grupos de personas.
- extraction_text = fragmento exacto de la medida o vínculo.
- attributes posibles:
   - TIPO: "prohibicion_acercamiento", "cese_perturbacion", "orden_cautelar", "contacto_prohibido", "restriccion_perimetral".
   - SUJETO_ACTIVO_GROUP: id del acusado/imputado.
   - SUJETO_PASIVO_GROUP: id de la víctima/denunciante/damnificado.
   - PLAZO: duración si está explícita (ejemplo: "6_meses").
   - DISTANCIA_MIN_M: distancia mínima en metros si está explícita (ejemplo: 500).
   - LUGARES_ALCANZADOS: direcciones o expresiones exactas como "cualquier lugar donde se encuentre la víctima".

CLASES:
- BANCO: entidad bancaria (no vale solo “Banco”).
- CBU: número de 22 dígitos.
- CORREO_ELECTRONICO: email (no el del juzgado).
- CUIT_CUIL: ##-########-#.
- CUIJ: código judicial ##-########-#.
- DIRECCION: domicilio en cualquiera de las formas listadas arriba.
- DNI: 7-8 dígitos o ##.###.### (no solo la palabra “DNI”).
- EDAD: edad en años o meses.
- ESTUDIOS: nivel educativo (primario, secundario, terciario, universitario, posgrado, doctorado; puede incluir “incompleto”, “completo”, “finalizado”, “en curso”).
- FECHA: fechas explícitas (excepto la de resolución del documento, y no expresiones vagas como “ayer” o “a las 15:00 horas”).
- LINK: URL.
- LOC: localidad, provincia, país, continente (no hospitales ni referencias genéricas).
- MARCA_AUTOMOVIL: marca de vehículo.
- NACIONALIDAD: nacionalidad.
- NUM_CAJA_AHORRO: número de caja de ahorro/cuenta.
- NUM_EXPEDIENTE: \d+/\d{4}.
- NUM_MATRICULA: matrícula profesional o académica.
- PATENTE_DOMINIO: dominio de vehículo (AAA123 o AA123AA).
- PER: nombre completo, iniciales o apodo de persona física.
- NUM_ACTUACION: número de actuación administrativa/contravencional.
- TELEFONO: fijo o celular.
- RELACION: vínculo o medida entre personas (ver atributos arriba).
""")


In [ ]:
# Si quisieramos excluir direcciones o ciertos campos agregar al prompt por ejemplo:
excepcion_prompt_comment = """- No extraigas:
   1) La fecha y lugar de la resolución del documento (ejemplo: "Buenos Aires, 29 de julio de 2022"), que suele aparecer al comienzo.
   2) Información del juzgado como mail, dirección, teléfono o redes sociales (ejemplo: "Juzgado PCyF No 10 - Tacuarí 138, 7o Piso - juzcyf10@jusbaires.gob.ar - 4014-6821/20 - @jpcyf10"), que suele estar al pie del documento.
"""

In [ ]:
NO_ATTRIBUTES_PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles para anonimización en documentos judiciales en español. Vas a leer cada parrafo con atención y extraer todas las entidades que correspondan según
las clases definidas más abajo. 

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como clase Persona, no confundir con siglas de otras cosas.
- Extraé SOLO spans EXACTOS que estén en el texto (no parafrasees ni infieras).
- Si una clase NO aparece, NO devuelvas nada de esa clase.
- NO inventes códigos ni números. No completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- NO son sensibles las siguientes entidades: 
                         1. La fecha y lugar de la resolucion del documento (por ej. "Buenos Aires, 29 de julio de 2022"), suele aparecen al comienzo del documento.
                         2. Información del juzgado como mail, direccion, telefono y cuenta de red social (por ej. "'Juzgado PCyF No 10 - Tacuarí 138, 7o Piso - juzcyf10@jusbaires.gob.ar - 4014-6821/20 - @jpcyf10'"), suele estar al pie del documento.
                         3. Las personas no sensibles como jueces, fiscales y secretarios.

- Si una entidad no está clara, NO la extraigas.
- Si una entidad está sutilmente mal escrita, incompleta, repetida o en un formato no estándar, pero detectas que corresponde a esa entendidad, extráela igual.

POSIBLES CLASES Y DESCRIPCIONES:
- BANCO: Debe especificar una entidad bancaria, 'Banco' no aplica como tal.
- CBU: número de 22 dígitos asociado a una entidad bancaria
- CORREO_ELECTRONICO: dirección de email, que no sea del juzgando.
- CUIT_CUIL: código único de identificación tributaria o laboral en Argentina (formato ##-########-#)
- CUIJ: código único de identificación judicial (formato ##-########-#)
- DIRECCION: puede presentarse como calle y altura (por ej. "9 de Julio 123"), intersección de calles (por ej. "Callao y Corrientes"), o número de domicilio 
- DNI: documento nacional de identidad de 7-8 dígitos, puede presentarse en el formato ##.###.### (numeros separados con puntos), la expresión "DNI" sin número no aplica como DNI.
- EDAD: edad de una persona, puede estar en años o meses
- ESTUDIOS: nivel educativo alcanzado (primario, secundario, terciario, universitario, posgrado, doctorado), puede estar acompañado de "incompleto", "completo", "finalizado", "en curso"
- FECHA: fechas en cualquier formato (dd/mm/aaaa, dd-mm-aaaa, dd de mes de aaaa, también puede ser dos fechas juntas como por ejemplo el 5 y 7 de mayo de 2020 y similares). No asignar como FECHA la fecha de resolución del documento, tampoco expresiones de fechas que no refieren a una fecha en particular, como por ejemplo, "en el día de ayer" o "a las 15:00 horas"
- LINK: URLs o enlaces web.
- LOC: nombres de localidades, provincias, países, continentes. NO asignar como LOC a lugares que no sean locaciones, como ser hospitales o referencias a lugares por nombres como "el domicilio de Olavarria"
- MARCA_AUTOMOVIL: marcas de automóviles (Ford, Chevrolet, Toyota, Renault, Fiat, etc)
- NACIONALIDAD: nacionalidades (argentina, italiana, española, uruguaya, chilena, paraguara, etc)
- NUM_CAJA_AHORRO: número de caja de ahorro o cuenta bancaria
- NUM_EXPEDIENTE: número de expediente judicial o administrativo en formato \d+/\d{4} (por ejemplo 1234/2020)
- NUM_MATRICULA: número de matrícula profesional (médica, abogacía, etc) o académica.
- PATENTE_DOMINIO: patentes o dominio de un vehículo. En Argentina, pueden ser de formato [A-Z]{3}\d{3} o [A-Z]{2}\d{3}[A-Z]{2}
- PER: Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.                     
- NUM_ACTUACION: Número identificatorio de una actuación administrativa o contravencional.
- TELEFONO: Número telefónico (fijo o celular).

""")


In [22]:
examples = [

# 1) PER/FECHA/DIRECCION/LOC (carátula + encabezado + dirección + fecha)
lx.data.ExampleData(
    text=textwrap.dedent("""JUZGADO DE FAMILIA N.º 1 DE LA CIUDAD DE SAN LORENZO
Expediente N.º 3187/2023
Carátula: Rodríguez, Carla Carolina c/ Fernández, Carlos Esteban s/ Violencia Familiar
SENTENCIA
En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, la Sra. Jueza Dra. Verónica Perez dicta resolución.
Con fecha 10 de noviembre de 2023, la Sra. Carla Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, San Lorenzo, denunció al Sr. Carlos Esteban Fernández, DNI 31.998.210."""),
    extractions=[
        lx.data.Extraction(extraction_class="PER", extraction_text="Rodríguez, Carla Carolina", group_index=0),
        lx.data.Extraction(extraction_class="PER", extraction_text="Fernández, Carlos Esteban", group_index=1),
        lx.data.Extraction(extraction_class="LOC",  extraction_text="CIUDAD DE SAN LORENZO"),
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="3187/2023"),
        lx.data.Extraction(extraction_class="LOC",  extraction_text="San Lorenzo"),
        lx.data.Extraction(extraction_class="LOC",  extraction_text="Provincia de Santa Fe"),
        lx.data.Extraction(extraction_class="PER",  extraction_text="Verónica Perez", group_index=2, attributes={"ROL_PROCESAL":"Juez"}),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="22 días del mes de noviembre de 2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="10 de noviembre de 2023"),
        lx.data.Extraction(extraction_class="PER",       extraction_text="Carla Carolina Rodríguez", group_index=0),
        lx.data.Extraction(extraction_class="DNI",       extraction_text="34.112.456",            group_index=0),
        lx.data.Extraction(extraction_class="DIRECCION", extraction_text="calle Belgrano 785, San Lorenzo", group_index=0),
        lx.data.Extraction(extraction_class="PER", extraction_text="Carlos Esteban Fernández", group_index=1),
        lx.data.Extraction(extraction_class="DNI", extraction_text="31.998.210",             group_index=1),
    ],
),

# 2) Único ejemplo de códigos/formatos + vehículo en narración
lx.data.ExampleData(
    text=textwrap.dedent("""CUIJ: IPP J-01-00017381-5/2021-0
Actuación Nro: 14544192/2021
Expte. N.o 4533/2024
El imputado se retiró manejando un vehículo Volkswagen Voyage, dominio KXY-876, por la Av. Corrientes. El video pertinente a la causa se encuentra disponible en https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv, grabado el 4 de abril de 2021"""),
    extractions=[
        lx.data.Extraction(extraction_class="CUIJ",              extraction_text="IPP J-01-00017381-5/2021-0"),
        lx.data.Extraction(extraction_class="NUM_ACTUACION",     extraction_text="14544192/2021"),
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE",    extraction_text="4533/2024"),
        lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL",   extraction_text="Volkswagen Voyage"),
        lx.data.Extraction(extraction_class="PATENTE_DOMINIO",   extraction_text="KXY-876"),
        lx.data.Extraction(extraction_class="LINK",              extraction_text="https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"),
        lx.data.Extraction(extraction_class="FECHA",             extraction_text="4 de abril de 2021"),
    ],
),

# 3) Datos personales típicos + otro vehículo/patente en frase fluida
lx.data.ExampleData(
    text=textwrap.dedent("""El Fiscal Carlos Garcia solicitó medidas respecto de ROBERTO CARUZO, DNI 30.112.642, 34 años de edad, nacionalidad paraguaya, con estudios secundarios completos, teléfono 1141504528 y correo electrónico rob.caruzo@gmail.com. 
En ese momento conducía un Ford Focus AA123BB por la Av. Rivadavia, en dirección a Caballito. El acusado violentó a Maria Garcia, de 25 años y DNI 45091234"""),
    extractions=[
        lx.data.Extraction(extraction_class="PER", extraction_text="Carlos Garcia", attributes={"ROL_PROCESAL":"Fiscal"}, group_index=0),
        lx.data.Extraction(extraction_class="PER",              extraction_text="ROBERTO CARUZO",           group_index=1, attributes={"ROL_PROCESAL": "Acusado"}),
        lx.data.Extraction(extraction_class="DNI",              extraction_text="30.112.642",               group_index=1),
        lx.data.Extraction(extraction_class="EDAD",             extraction_text="34",                       group_index=1),
        lx.data.Extraction(extraction_class="NACIONALIDAD",     extraction_text="paraguaya",                group_index=1),
        lx.data.Extraction(extraction_class="ESTUDIOS",         extraction_text="estudios secundarios completos", group_index=1),
        lx.data.Extraction(extraction_class="TELEFONO",         extraction_text="1141504528",               group_index=1),
        lx.data.Extraction(extraction_class="CORREO_ELECTRONICO", extraction_text="rob.caruzo@gmail.com",  group_index=1),
        lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL",  extraction_text="Ford Focus",               group_index=1),
        lx.data.Extraction(extraction_class="PATENTE_DOMINIO",  extraction_text="AA123BB",                  group_index=1),
        lx.data.Extraction(extraction_class="PER",              extraction_text="Maria Garcia",               group_index=3, attributes={"ROL_PROCESAL": "Victima"}),
        lx.data.Extraction(extraction_class="DNI",              extraction_text="45091234",               group_index=3),
        lx.data.Extraction(extraction_class="EDAD",             extraction_text="25",                       group_index=3),       
    ],
),

# 4) NEGATIVO
lx.data.ExampleData(
    text=textwrap.dedent("""VISTOS: Que a fin de ordenar la marcha del proceso, se fija audiencia preliminar. No se consignan números de expediente, CUIJ ni domicilios en el presente proveído."""),
    extractions=[],
),
]


In [19]:
# -----------------
# Ejemplos balanceados (judiciales)
#   1) PER/FECHA/DIRECCION/LOC
#   2) Un único ejemplo de códigos (para enseñar formato)
#   3) Datos personales típicos de actuaciones
#   4) NEGATIVO: no hay códigos -> salida vacía
# -----------------
long_examples = [

lx.data.ExampleData(
    text=textwrap.dedent("""JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar
SENTENCIA
En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados "Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar", Expte. N.o 3187/2023.
I. ANTECEDENTES
Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial.
La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado comenzó a aislarla de su entorno familiar, controlar sus gastos y proferir insultos y amenazas, que en los últimos meses derivaron en empujones y golpes.
Se acompañaron constancias médicas del Hospital "San Martín" de fechas 14/08/2023 y 08/11/2023, donde se registran lesiones en antebrazos y región lumbar, así como un informe psicológico que describe un cuadro de depresión moderada y ansiedad.
II. MEDIDAS CAUTELARES ADOPTADAS
Con fecha 11 de noviembre de 2023, este Juzgado dispuso:
Exclusión inmediata del denunciado del domicilio conyugal.
Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros.
Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales.
III. PRUEBA PRODUCIDA
En audiencia celebrada el 17 de noviembre de 2023, declararon la Sra. Patricia Gómez, vecina del domicilio conyugal, y el Sr. Luis Alberto Rivas, compañero de trabajo de la denunciante, quienes relataron haber presenciado episodios de gritos, discusiones y amenazas por parte del denunciado.
El Equipo Interdisciplinario del Juzgado emitió informe en el que concluyó que existe un riesgo alto de reiteración de la violencia, con impacto negativo en el bienestar emocional de la menor.
IV. FUNDAMENTOS
De la valoración integral de la prueba surge acreditada la existencia de violencia física, psicológica y patrimonial ejercida por el Sr. Diego Esteban Fernández contra la Sra. Ana Carolina Rodríguez, en el marco de una relación de pareja y con afectación a la hija menor.
La Ley Nacional 26.485 y la Ley Provincial 11.529 obligan a adoptar medidas urgentes y eficaces para proteger a las víctimas de violencia de género.
En este caso, la reiteración de los hechos, la proximidad de los domicilios y la vulnerabilidad de la menor justifican la extensión de las medidas cautelares y la adopción de acciones complementarias.
V. RESUELVO
Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la Sra. Ana Carolina Rodríguez y de la hija menor M.F.R.
Mantener la exclusión del denunciado del domicilio conyugal. La grabación de la sentencia se encuentra disponible en el link: https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv. """),
    extractions=[
        # Personas y atributos
        # Carátula: nombres invertidos
        lx.data.Extraction(extraction_class="PER", extraction_text="Rodríguez, Ana Carolina", group_index=0, attributes={"ROL_PROCESAL":"Denunciante"}),
        lx.data.Extraction(extraction_class="PER", extraction_text="Fernández, Diego Esteban", group_index=1),

        # Repetición de expediente
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="3187/2023"),

        # Relación de cónyuge / atributo
        lx.data.Extraction(extraction_class="PER", extraction_text="Diego Esteban Fernández", group_index=1, attributes={"RELACION_CON_VICTIMA":"cónyuge"}),

        # Relación de hija
        lx.data.Extraction(extraction_class="PER", extraction_text="M.F.R.", group_index=2, attributes={"RELACION_CON_VICTIMA":"hija"}),

        # Roles de testigos más contexto
        lx.data.Extraction(extraction_class="PER", extraction_text="Patricia Gómez", group_index=4, attributes={"ROL_PROCESAL":"Testigo","RELACION_CON_VICTIMA":"vecina"}),
        lx.data.Extraction(extraction_class="PER", extraction_text="Luis Alberto Rivas", group_index=5, attributes={"ROL_PROCESAL":"Testigo","RELACION_CON_VICTIMA":"compañero de trabajo"}),
        lx.data.Extraction(extraction_class="PER", extraction_text="Ana Carolina Rodríguez", group_index=0, attributes={"ROL_PROCESAL":"Denunciante"}),
        lx.data.Extraction(extraction_class="DNI", extraction_text="34.112.456", group_index=0),
        lx.data.Extraction(extraction_class="DIRECCION", extraction_text="calle Belgrano 785, Barrio Centro, San Lorenzo", group_index=0, attributes={"TIPO":"domicilio_de_la_victima"}),

        lx.data.Extraction(extraction_class="PER", extraction_text="Diego Esteban Fernández", group_index=1),
        lx.data.Extraction(extraction_class="DNI", extraction_text="31.998.210", group_index=1),
        lx.data.Extraction(extraction_class="DIRECCION", extraction_text="calle Moreno 1520", group_index=1, attributes={"TIPO":"domicilio_del_imputado"}),

        lx.data.Extraction(extraction_class="PER", extraction_text="M.F.R.", group_index=2),
        lx.data.Extraction(extraction_class="EDAD", extraction_text="9 años", group_index=2),

        lx.data.Extraction(extraction_class="PER", extraction_text="Verónica Salvatierra", group_index=3, attributes={"ROL_PROCESAL":"Juez"}),

        lx.data.Extraction(extraction_class="PER", extraction_text="Patricia Gómez", group_index=4, attributes={"ROL_PROCESAL":"Testigo"}),
        lx.data.Extraction(extraction_class="PER", extraction_text="Luis Alberto Rivas", group_index=5, attributes={"ROL_PROCESAL":"Testigo"}),

        # Localidades
        lx.data.Extraction(extraction_class="LOC", extraction_text="San Lorenzo"),
        lx.data.Extraction(extraction_class="LOC", extraction_text="Provincia de Santa Fe"),

        # Expediente
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="3187/2023"),

        # Fechas (excluye la de resolución)
        lx.data.Extraction(extraction_class="FECHA", extraction_text="10 de noviembre de 2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="14/08/2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="08/11/2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="11 de noviembre de 2023"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="17 de noviembre de 2023"),

        #Link
        lx.data.Extraction(extraction_class="LINK", extraction_text="https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"),

        # Relaciones (medidas cautelares y vínculos)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros.",
            attributes={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0,"DISTANCIA_MIN_M":500}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prohibición de acercamiento a la denunciante y a la hija menor en un radio de 500 metros.",
            attributes={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":2,"DISTANCIA_MIN_M":500}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales.",
            attributes={"TIPO":"contacto_prohibido","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prohibición de contacto por cualquier medio, incluidas llamadas, mensajes y redes sociales.",
            attributes={"TIPO":"contacto_prohibido","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":2}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Exclusión inmediata del denunciado del domicilio conyugal.",
            attributes={"TIPO":"orden_cautelar","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la Sra. Ana Carolina Rodríguez",
            attributes={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0,"PLAZO":"180_dias"}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="Prorrogar por el plazo de 180 días la prohibición de acercamiento y de contacto del Sr. Diego Esteban Fernández respecto de la hija menor M.F.R.",
            attributes={"TIPO":"prohibicion_acercamiento","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":2,"PLAZO":"180_dias"}
        ),

        # Vínculo conyugal / convivencial (entre personas)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="su cónyuge",
            attributes={"TIPO":"vinculo_conyugal","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":0}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="conviven desde hace doce años",
            attributes={"TIPO":"vinculo_convivencia","SUJETO_ACTIVO_GROUP":0,"SUJETO_PASIVO_GROUP":1}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="tienen una hija en común, M.F.R.",
            attributes={"TIPO":"vinculo_filiacion","SUJETO_ACTIVO_GROUP":0,"SUJETO_PASIVO_GROUP":2}
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="tienen una hija en común, M.F.R.",
            attributes={"TIPO":"vinculo_filiacion","SUJETO_ACTIVO_GROUP":1,"SUJETO_PASIVO_GROUP":2}
        ),

    ],
),
lx.data.ExampleData(
    text=textwrap.dedent("""
        JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19
        GOMEZ, ELVIS JUNIOR SOBRE 89 - LESIONES LEVES
        Número: IPP 8125/2020-0
        CUIJ: IPP J-01-00017381-5/2021-0
        Actuación Nro: 14544192/2021
        ACTA DE AUDIENCIA
        VIDEOCONFERENCIA
        "GOMEZ, ELVIS JUNIOR SOBRE 89 EN FUNCIÓN DEL 92, 149 BIS, 162 Y 239 DEL CÓDIGO PENAL"
        Causa N° 8122/2021
        Fecha: 4 de abril de 2021
        Horario de inicio: 12:00 horas
        Tipo de audiencia: audiencia de conocimiento personal (art. 266 CPPCABA)
        Juez: Pablo C. Casas -Juzgado Penal Contravencional y de Faltas Nro. 10-.
        Secretaria: Maria Agustina Iriarte López.
        PARTES PRESENTES
        Acusado: Carlos Junior PEREZ, DNI n° 50.966.533.
        Defensa Oficial: Marina Recabarra, -Defensoría Oficial Nro. 20-.
        Fiscal: Adrián Dávila -Fiscalía Penal, Contravencional y de Faltas Nro. 36-.
        DESARROLLO
        El 15 de marzo de 2021, alrededor de las 23:00 horas, mientras se encontraba en una reunión en la casa de una señora llamada SOL,
        en Villa Pueyrredón de esta ciudad, se puso agresivo con su pareja BELEN GUTIERREZ, le pegó dos piñas en la cara,
        la agarró del cuello y la tiró al piso. Luego procedió a llevarse las llaves de su vehículo Volkswagen Voyage, dominio KXY-876."""),
            
    extractions=[
        # Identificadores judiciales
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="8122/2021"),
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="8125/2020"),
        lx.data.Extraction(extraction_class="CUIT_CUIL", extraction_text="IPP J-01-00017381-5/2021-0"),
        lx.data.Extraction(extraction_class="NUM_ACTUACION", extraction_text="14544192/2021"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="4 de abril de 2021"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="15 de marzo de 2021"),
        lx.data.Extraction(extraction_class="LOC", extraction_text="Villa Pueyrredón"),

        # Funcionario judicial
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Pablo C. Casas",
            attributes={"ROL_PROCESAL": "Juez"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Maria Agustina Iriarte López",
            attributes={"ROL_PROCESAL": "Secretario"},
            group_index=2
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Adrián Dávila",
            attributes={"ROL_PROCESAL": "Fiscal"},
            group_index=3
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Marina Recabarra",
            attributes={"ROL_PROCESAL": "Defensor Oficial"},
            group_index=4
        ),

        # Acusado
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Carlos Junior PEREZ",
            attributes={"ROL_PROCESAL": "Acusado"},
            group_index=5
        ),
        lx.data.Extraction(
            extraction_class="DNI",
            extraction_text="50.966.533",
            group_index=5
        ),

        # Víctima
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="BELEN GUTIERREZ",
            attributes={"ROL_PROCESAL": "Victima"},
            group_index=6
        ),
        lx.data.Extraction(extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876",group_index=6),
        lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL", extraction_text="Volkswagen Voyage",group_index=6),
        # Testigo circunstancial (persona mencionada, no procesal)
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="SOL",
            attributes={"ROL_PROCESAL": "Testigo"},
            group_index=7
        ),

        # Relación entre acusado y víctima (violencia física)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="le pegó dos piñas en la cara, la agarró del cuello y la tiró al piso",
            attributes={
                "TIPO": "cese_perturbacion",   # violencia física → medida/restricción
                "SUJETO_ACTIVO_GROUP": 5,
                "SUJETO_PASIVO_GROUP": 6
            }
        )
    ],
),
lx.data.ExampleData(
    text=textwrap.dedent("""
        Hoy el Fiscal Carlos Garcia solicitó que se ordene a ROBERTO CARUZO, DNI 30.112.642, 34 años de edad, nacionalidad paraguaya, con 
        estudios secundarios completos, por el plazo de seis meses, el cese en los actos de perturbación o intimidación que, directa o indirectamente, 
        realice hacia la persona de la damnificada BELEN CASIO, CUIL 27-37.412.987-6; 2) por el plazo de seis (6) meses, se imponga a ROBERTO CARUZO, DNI 30.112.642
        la prohibición de acercamiento y contacto hacia la víctima BELEN CASIO, 37.412.987, –en el lugar que se encuentre– de
        modo que deberá suspender todo tipo de contacto físico y/o por cualquier medio que signifique intromisión injustificada
        en relación a la persona de la damnificada, por sí o por intermedio de terceras personas y la prohibición de acercamiento
        a menos de 500 metros de los domicilios ubicados en Av. Pedro Goyena 51, piso 7° dpto. "A", y Callao 543, de esta Ciudad. Se requirió  informes  al  
        Banco BBVA   Francés, respecto  de  las  cuentas  bancarias  del denunciado ROBERTO CARUZO, identificadas  como  Caja  de  ahorro  en  pesos  argentinos 
        número  117-59824/6 con  CBU  0180132640000004685591, teléfono celular 1141504528 y dirección de correo electrónico rob.caruzo@gmail.com."""),
    extractions=[

        # Fiscal (grupo 0: funcionario, si querés diferenciarlo)
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Carlos Garcia",
            attributes={"ROL_PROCESAL": "Fiscal"},
            group_index=0
        ),
        # Víctima
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="BELEN CASIO",
            attributes={"ROL_PROCESAL": "Victima"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="CUIL",
            extraction_text="27-37.412.987-6",
            group_index=1
        ),
        # Direcciones de la víctima (ambas al grupo 1)
        lx.data.Extraction(
            extraction_class="DIRECCION",
            extraction_text="Av. Pedro Goyena 51, piso 7° dpto. \"A\"",
            attributes={"TIPO": "domicilio_de_la_victima"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="DIRECCION",
            extraction_text="Callao 543",
            attributes={"TIPO": "domicilio_de_la_victima"},
            group_index=1
        ),

        # Acusado / Imputado
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="ROBERTO CARUZO",
            attributes={"ROL_PROCESAL": "Acusado"},
            group_index=2
        ),
        lx.data.Extraction(
            extraction_class="DNI",
            extraction_text="30.112.642",
            group_index=2
        ),
        lx.data.Extraction(extraction_class="BANCO",         extraction_text="Banco BBVA", group_index=2),
        lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6", group_index=2),
        lx.data.Extraction(extraction_class="CBU",        extraction_text="0180132640000004685591", group_index=2),
        lx.data.Extraction(extraction_class="TELEFONO",     extraction_text="1141504528",group_index=2),
        lx.data.Extraction(extraction_class="CORREO_ELECTRONICO", extraction_text="rob.caruzo@gmail.com",group_index=2),
        lx.data.Extraction(extraction_class="EDAD",        extraction_text="34", group_index=2),
        lx.data.Extraction(extraction_class="NACIONALIDAD",extraction_text="paraguaya", group_index=2),
        lx.data.Extraction(extraction_class="ESTUDIOS",    extraction_text="estudios secundarios completos", group_index=2),

        # Relación / Medidas (conecta grupos 2 -> 1)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="cese en los actos de perturbación o intimidación por el plazo de seis meses",
            attributes={
                "TIPO": "cese_perturbacion",
                "SUJETO_ACTIVO_GROUP": 2,
                "SUJETO_PASIVO_GROUP": 1,
                "PLAZO": "6_meses"
            }
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="prohibición de acercamiento y contacto por el plazo de seis (6) meses a menos de 500 metros",
            attributes={
                "TIPO": "prohibicion_acercamiento",
                "SUJETO_ACTIVO_GROUP": 2,
                "SUJETO_PASIVO_GROUP": 1,
                "PLAZO": "6_meses",
                "DISTANCIA_MIN_M": 500,
                "LUGARES_ALCANZADOS": [
                    "Av. Pedro Goyena 51, piso 7° dpto. \"A\"",
                    "Callao 543",
                    "cualquier lugar donde se encuentre la víctima"
                ]
            }
        ),
    ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""VISTOS: Que a fin de ordenar la marcha del proceso, se fija audiencia preliminar. "
              "No se consignan números de expediente, CUIJ ni domicilios en el presente proveído."""),
        extractions=[],  # ejemplo negativo: desalienta devolver clases ausentes
    ),
]

## Functions

In [39]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

def take_start_end_paragraphs(paragraphs):
    # Create start and end character positions
    start_end_chars = []
    current_pos = 0

    for i, paragraph in enumerate(paragraphs):
        start_char = current_pos
        end_char = start_char + len(paragraph)
        start_end_chars.append(
            {
                "paragraph_position": i,
                "text": paragraph,
                "paragraph_id": str(text_to_uuid(paragraph)).replace("-", ""),
                "start_char": start_char,
                "end_char": end_char,
            }
        )
        # +1 for the newline character between paragraphs (except after the last one)
        current_pos = end_char + 1

    return start_end_chars

def constract_paragraph(document):
    paragraphs = [line.strip() for line in document.split("\n") if line.strip()]
    paragraphs = [re.sub(r"\s{2,}", " ", line) for line in paragraphs]
    paragraphs = list(unique_justseen(paragraphs))
    return paragraphs

def langextract_to_dict(result):
    out = []
    for i in range(len(result.extractions)):
        #paragraph = result.text
        label = result.extractions[i].extraction_class
        text = result.extractions[i].extraction_text
        start_char = result.extractions[i].char_interval.start_pos
        end_char = result.extractions[i].char_interval.end_pos
        attrs = result.extractions[i].attributes
        alignment_status = result.extractions[i].alignment_status

        out.append({
            "label": label,
            "text": text,
            "start_char": start_char,
            "end_char": end_char,
            "attrs": attrs,
            "alignment_status": alignment_status
        })
        #print(f"{i+1}: \n CLASS: {result.extractions[i].extraction_class} \n TEXT: {result.extractions[i].extraction_text} \n from_chars: {text[result.extractions[i].char_interval.start_pos:result.extractions[i].char_interval.end_pos]}")
    return out

def langextract_prediction(text,PROMPT, examples,openai_api_key):

    result =  lx.extract(
        text_or_documents=text,
        prompt_description=PROMPT,
        examples=examples,
        language_model_type=lx.inference.OpenAILanguageModel,
        model_id="gpt-4o",
        api_key=openai_api_key,
        max_char_buffer=1000, #1000,
        extraction_passes=2, #1,
        max_workers=4, #6,
        fence_output=True,
        use_schema_constraints=False, # https://github.com/google/langextract
        language_model_params={
            "temperature": 0.0, #0.1,
            "top_p": 1.0,
            "max_tokens": 400,
            "timeout": 600,},
            debug=False)
    return result, langextract_to_dict(result)

def sample_cases(df, label, model="prediction", n=3):
    # nombre dinámico para el campo de predicción principal
    pred_key = "openai" if model == "prediction" else ("ner" if model == "NER_prediction" else "predmodel")

    def _norm_label_text(item):
        if not isinstance(item, dict):
            return None, ""
        attrs = item.get("attrs", {}) or {}
        lab   = attrs.get("aymurai_label") or item.get("label")
        txt   = attrs.get("aymurai_alt_text") or item.get("extraction_text") or item.get("text", "")
        return lab, txt

    cases = {"TP": [], "FP": [], "FN": []}

    for _, row in df.iterrows():
        # validation
        val_raw = eval(row["validation"]) if row["validation"] else []
        val_spans = [_norm_label_text(v) for v in val_raw if isinstance(v, dict)]
        val_spans = [(lab, txt) for lab, txt in val_spans if lab is not None]

        # pred principal (según 'model')
        if model == "NER_prediction":
            pred_raw = eval(row[model]) if row.get(model) else []
        else:
            pred_raw = row.get(model) if row.get(model) else []
        pred_spans = [_norm_label_text(p) for p in pred_raw if isinstance(p, dict)]
        pred_spans = [(lab, txt) for lab, txt in pred_spans if lab is not None]

        # pred de ambos modelos (si existen las columnas)
        openai_raw = row.get("prediction")
        ner_raw    = row.get("NER_prediction")
        openai_list = openai_raw if (openai_raw and not isinstance(openai_raw, str)) else (eval(openai_raw) if openai_raw else [])
        ner_list    = ner_raw if (ner_raw and not isinstance(ner_raw, str)) else (eval(ner_raw) if ner_raw else [])

        openai_spans = [_norm_label_text(p) for p in (openai_list or []) if isinstance(p, dict)]
        openai_spans = [(lab, txt) for lab, txt in openai_spans if lab is not None]
        ner_spans    = [_norm_label_text(p) for p in (ner_list or []) if isinstance(p, dict)]
        ner_spans    = [(lab, txt) for lab, txt in ner_spans if lab is not None]

        # listas planas por label
        val_labels  = [lab for lab, _ in val_spans]
        pred_labels = [lab for lab, _ in pred_spans]

        # snippet e info extra
        text     = (row.get("text_x") or "")#[:300]
        para_id  = row.get("paragraph_id", None)
        doc_name = row.get("name", None)

        # armar registro con val + ambas preds + clave dinámica
        base_rec = {
            "paragraph_id": para_id,
            "document": doc_name,
            "parrafo": text,
            "val":   [t for lab, t in val_spans if lab == label],
            "openai": [t for lab, t in openai_spans if lab == label],
            "ner":    [t for lab, t in ner_spans    if lab == label],
        }
        # setear pred principal bajo su clave dinámica
        base_rec[pred_key] = [t for lab, t in pred_spans if lab == label]

        # clasificar TP/FP/FN
        if (label in val_labels) and (label in pred_labels):
            cases["TP"].append(base_rec)
        elif (label in val_labels) and (label not in pred_labels):
            cases["FN"].append(base_rec)
        elif (label not in val_labels) and (label in pred_labels):
            cases["FP"].append(base_rec)

    # muestra aleatoria top-n por tipo
    return {k: random.sample(v, min(len(v), n)) for k, v in cases.items() if v}


In [29]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]
docs_file

['document-06.docx',
 'documento-entrerios-03.docx',
 'documento-entrerios-02.docx',
 'document-07.docx',
 'document-02.docx',
 'document-03.docx',
 'document-04.docx',
 'document-08.docx']

In [30]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]

docs_file
documents = {}
doc_paragraphs = {}
doc_start_end_chars = {}
joined_texts = {}
for d in docs_file:
    path = DOCS_PATH + d
    # Extract document
    document = extraction(path)
    # Construct paragraphs
    paragraphs = constract_paragraph(document)
    start_end_chars = take_start_end_paragraphs(paragraphs)
    documents[d] = document
    doc_paragraphs[d] = paragraphs
    joined_text = "\n".join(paragraphs)
    joined_texts[d] = joined_text
    doc_start_end_chars[d] = start_end_chars


In [ ]:
joined_texts

In [ ]:
pprint(examples)

In [40]:
import time

start = timeit.timeit()
predictions = {}
results = {}

name_doc = 'documento-entrerios-02.docx'
text = joined_texts[name_doc]

results[name_doc], predictions[name_doc] = langextract_prediction(text, PROMPT, examples, openai_api_key)

# for name_doc, text in joined_texts.items():
#     print(name_doc)
#     results[name_doc], predictions[name_doc] = langextract_prediction(text, PROMPT, examples, openai_api_key)
#     time.sleep(25)  # Sleep for 2 seconds to avoid rate limit

end = timeit.timeit()
print('\n Total time: ', end-start)


/var/folders/yf/2cttyjmx5j35yhq4st33mdmm0000gn/T/ipykernel_4196/2750225199.py:56: DeprecationWarning: 'language_model_type' is deprecated and will be removed in v2.0.0. Use model, config, or model_id parameters instead.
  result =  lx.extract(


[12:28:52] INFO     Starting sequential extraction passes for improved recall with 2 passes.      ]8;id=296463;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=268010;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#404\404]8;;\

           INFO     Starting extraction pass 1 of 2                                               ]8;id=403016;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=968831;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#416\416]8;;\

           INFO     Starting document annotation.                                                 ]8;id=938129;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=181550;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#261\261]8;;\

           INFO     Processing batch 0 with length 4                                              ]8;id=707874;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=15905;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#282\282]8;;\

...

           INFO     Retrying request to /chat/completions in 0.378995 seconds                  ]8;id=930784;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/openai/_base_client.py\_base_client.py]8;;\:]8;id=444840;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/openai/_base_client.py#1071\1071]8;;\

[12:28:57] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too ]8;id=927167;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=333917;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\
                    Many Requests"                                                                                 

           INFO     Retrying request to /chat/completions in 0.784667 seconds                  ]8;id=260172;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/openai/_base_client.py\_base_client.py]8;;\:]8;id=844048;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/openai/_base_client.py#1071\1071]8;;\

[12:28:58] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too ]8;id=441302;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=994211;file:///Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/.venv/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\
                    Many Requests"                                                                                 

InferenceRuntimeError: Parallel inference error: OpenAI API error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [13]:
import pickle

result_path = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/notebooks/experiments/anonymization/results_openai-attributes-entrerios0203.pkl'

with open(result_path,"rb") as file:
    results = pickle.load(file)

In [ ]:
pprint(results['documento-entrerios-02.docx'])